# Verifica etichetta anno per il lead-year combinato stagionale (DJF)

`seasonal_ACC_calculation.ipynb` mostra un segnale negativo anomalo e molto
forte nella regione ENSO nelle mappe di differenza di ACC (SENS-CTRL). La
diagnostica sui timestamp (vedi cella dedicata in quel notebook) ha escluso
l'ipotesi piu' semplice (un salto di un anno per timestamp di gennaio
riportati a dicembre): i file grezzi del modello etichettano ogni anno con un
placeholder a **giugno**, mai gennaio, e `.replace(month=12)` lo sposta a
dicembre mantenendo lo stesso anno - comportamento corroborato da un blocco
commentato in `05-Serie_temporale_anomalie.ipynb` (DJF e SON: nessun offset
d'anno; solo MAM/JJA richiedono +1 anno).

Resta pero' una domanda che i soli timestamp non possono risolvere: **"giugno
Y" e' l'anno di INIZIALIZZAZIONE della hindcast, o l'anno di VERIFICA
(target)?** Per il lead "3-4" (anni di lead 3 e 4 dopo l'inizializzazione), se
fosse l'anno di inizializzazione, la finestra DJF verificata sarebbe intorno a
dicembre Y+3/Y+4, non dicembre Y - un disallineamento di 3-4 anni che
`xr.align`/`.sel(time=slice(...))` non farebbe mai fallire con un errore
(l'obs copre comunque un range che si sovrappone "per caso"), correlando
sistematicamente anni sbagliati tra loro.

**Verifica fisica indipendente dai timestamp**: l'ENSO e' il segnale
interannuale piu' forte e meglio documentato nel record 1994-2018. Se la
serie CTRL/SENS di `tas` mediata sul box Nino3.4 (5S-5N, 170W-120W) e'
etichettata con l'anno corretto, deve mostrare un picco caldo netto nel 1997
e nel 2015 (i due El Nino piu' forti del periodo) e un segnale freddo intorno
al 1998-2000, 2007-08, 2010-11, 2020-22 (La Nina), in fase con l'ERA5
osservato. Se l'etichetta e' sbagliata di N anni, il modo piu' diretto per
scoprirlo (senza fidarsi dell'occhio) e' scorrere un ventaglio di traslazioni
temporali (lag, in anni) tra la serie del modello e quella osservata, e
vedere quale lag massimizza la correlazione: lag=0 conferma l'etichetta
attuale, un lag != 0 rivela l'errore e la sua entita'.


In [ ]:
# rende config.py (in notebooks/) importabile anche da questa sottocartella
import sys, os
_cfg = os.getcwd()
while _cfg != os.path.dirname(_cfg):
    if os.path.exists(os.path.join(_cfg, 'config.py')):
        sys.path.insert(0, _cfg)
        break
    _cfg = os.path.dirname(_cfg)
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy import stats

import albedo_functions as af

DATA_PATH = POST_DATA
OBS_PATH = WORK_DIR

exp_ctrl = 'a1ua'
exp_sens = 'a52o'
var = 'tas'
season = 'DJF'
y1, y2 = 3, 4          # stesso lead della mappa ACC segnalata come anomala
lead = f"{y1}-{y2}"
lead_number = y2 - y1 + 1

# Box Nino3.4 (5S-5N, 170W-120W), convenzione lon 0-360 come il resto della pipeline
NINO34_LAT_MIN, NINO34_LAT_MAX = -5, 5
NINO34_LON_MIN, NINO34_LON_MAX = 190, 240


In [ ]:
# Caricamento IDENTICO a seasonal_ACC_calculation.ipynb (stessa etichettatura
# in discussione: normalize+to_period('M'), poi .replace(month=12) per DJF).
# NON applico il taglio posizionale time[6] dell'originale: qui vogliamo il
# range completo per poter scorrere i lag senza assumere gia' che l'allineamento
# a lag=0 sia corretto (e' proprio quello che stiamo verificando).
dset_ctrl_path = f"{DATA_PATH}/{exp_ctrl}/1x1/{var}/{exp_ctrl}_{var}_Amon_EC-Earth3_dcppA-hindcast_lead_{lead}_1x1_ensemble_m{season}_rad.nc"
dset_ctrl = xr.open_dataset(dset_ctrl_path)
dset_ctrl['time'] = pd.to_datetime(dset_ctrl['time'].values).normalize().to_period('M').start_time

dset_sens_path = f"{DATA_PATH}/{exp_sens}/1x1/{var}/{exp_sens}_{var}_Amon_EC-Earth3_dcppA-hindcast_lead_{lead}_1x1_ensemble_m{season}_rad.nc"
dset_sens = xr.open_dataset(dset_sens_path)
dset_sens['time'] = pd.to_datetime(dset_sens['time'].values).normalize().to_period('M').start_time

if season == 'DJF':
    dset_ctrl['time'] = [pd.Timestamp(t).replace(month=12) for t in dset_ctrl['time'].values]
    dset_sens['time'] = [pd.Timestamp(t).replace(month=12) for t in dset_sens['time'].values]

obs_path = OBS_PATH / f"ERA5_tas_1x1_{lead_number}{season}.nc"
obs = xr.open_dataset(obs_path)
obs['time'] = pd.to_datetime(obs['time'].values).normalize().to_period('M').start_time
obs = obs.rename({'2t': 'tas'})

print(f"CTRL anni etichettati: {sorted(pd.to_datetime(dset_ctrl.time.values).year.tolist())}")
print(f"SENS anni etichettati: {sorted(pd.to_datetime(dset_sens.time.values).year.tolist())}")
print(f"OBS  anni etichettati: {sorted(pd.to_datetime(obs.time.values).year.tolist())}")


In [ ]:
# Media d'ensemble + media sul box Nino3.4 + anomalia (stessa area-weighted
# mean di af.domain_selection, gia' usata in tutta la pipeline).
ctrl_nino = af.domain_selection(dset_ctrl[var].mean('member'), NINO34_LAT_MIN, NINO34_LAT_MAX, NINO34_LON_MIN, NINO34_LON_MAX)
sens_nino = af.domain_selection(dset_sens[var].mean('member'), NINO34_LAT_MIN, NINO34_LAT_MAX, NINO34_LON_MIN, NINO34_LON_MAX)
obs_nino  = af.domain_selection(obs[var],                      NINO34_LAT_MIN, NINO34_LAT_MAX, NINO34_LON_MIN, NINO34_LON_MAX)

ctrl_nino_anom = (ctrl_nino - ctrl_nino.mean('time')).assign_coords(time=pd.to_datetime(ctrl_nino['time'].values).year)
sens_nino_anom = (sens_nino - sens_nino.mean('time')).assign_coords(time=pd.to_datetime(sens_nino['time'].values).year)
obs_nino_anom  = (obs_nino  - obs_nino.mean('time')).assign_coords(time=pd.to_datetime(obs_nino['time'].values).year)


In [ ]:
# --- Scansione dei lag: trasla l'anno etichetta del modello di 'lag' anni e
# correla con l'obs (allineamento sugli anni in comune dopo la traslazione).
# lag=0 = l'etichetta attuale e' corretta. lag=+N = il modello deve essere
# spostato N anni AVANTI per corrispondere all'obs (l'anno vero e' N anni
# dopo l'etichetta attuale - coerente con l'ipotesi "giugno Y = anno di
# inizializzazione", che per un lead 3-4 darebbe un lag atteso vicino a +3/+4).
def lag_scan(model_da, obs_da, lags=range(-6, 7)):
    rows = []
    for lag in lags:
        shifted = model_da.assign_coords(time=model_da['time'].values + lag)
        m, o = xr.align(shifted, obs_da, join='inner')
        n = m.sizes.get('time', 0)
        if n < 5:
            rows.append((lag, np.nan, np.nan, n))
            continue
        r, p = stats.pearsonr(m.values, o.values)
        rows.append((lag, r, p, n))
    return rows

for label, model_da in [('CTRL', ctrl_nino_anom), ('SENS', sens_nino_anom)]:
    print(f"--- {label} vs OBS, Nino3.4 {var} anomaly, lead {lead} {season} ---")
    rows = lag_scan(model_da, obs_nino_anom)
    best = max((r for r in rows if not np.isnan(r[1])), key=lambda r: r[1], default=None)
    for lag, r, p, n in rows:
        marker = "  <-- massimo" if best is not None and lag == best[0] else ""
        r_str = f"{r:+.3f}" if not np.isnan(r) else "  n/d"
        p_str = f"p={p:.3f}" if not np.isnan(p) else ""
        print(f"  lag={lag:+3d}  r={r_str}  {p_str}  (n={n} anni in comune){marker}")
    print()


In [ ]:
# --- Confronto visivo con gli anni ENSO noti (nessun lag applicato: mostra
# l'etichetta COSI' COM'E' oggi nel codice, lag=0) ---
EL_NINO_FORTI = [1997, 2015]   # El Nino molto forti nel periodo 1994-2018
LA_NINA_FORTI = [1999, 2007, 2010, 2020, 2021]  # La Nina forti/moderate

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(ctrl_nino_anom['time'].values, ctrl_nino_anom.values, 'o-', label='CTRL (a1ua)', color='tab:blue')
ax.plot(sens_nino_anom['time'].values, sens_nino_anom.values, 'o-', label='SENS (a52o)', color='tab:orange')
ax.plot(obs_nino_anom['time'].values, obs_nino_anom.values, 'ko-', label='ERA5 (obs)', linewidth=2)

for y in EL_NINO_FORTI:
    ax.axvline(y, color='red', alpha=0.3, linewidth=8, zorder=0)
for y in LA_NINA_FORTI:
    ax.axvline(y, color='blue', alpha=0.2, linewidth=8, zorder=0)

ax.axhline(0, color='gray', linewidth=0.8)
ax.set_xlabel('anno (etichetta attuale, lag=0)')
ax.set_ylabel(f'anomalia {var} su box Nino3.4 (K)')
ax.set_title(f'Nino3.4 {var} anomaly, lead {lead} {season} - bande rosse=El Nino forti, blu=La Nina forti')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/nino34_label_check_{var}_lead_{lead}_{season}.png", dpi=200, bbox_inches='tight')
